## Generate Standard Sheets for Specific Clusters

Contains the following information:
- Highlighted cluster UMAP
- Profile violin plot (growth parameters, recall, other metrics?)
- Profile timeseries
- Gene analysis (bar chart w/ singletons as other)
- GO enrichment analysis (bar chart)
- Recall Measurements (bar chart)

### Determining Inclusion

- First, generate all clusters at the lowest resolution
- Then, examine clustree representation and take interesting clusters with higher median recall

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr
import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sp
import sklearn as skl
import dask.dataframe as dd
import dask.array as da
import dask
import warnings
import copy
import random
from sklearn.metrics.pairwise import euclidean_distances,manhattan_distances,cosine_distances

from sklearn.metrics import silhouette_score
import scipy.stats
from sklearn.linear_model import LinearRegression
from sklearn.cluster import AffinityPropagation
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering

import scanpy as sc
import anndata
import scipy as sp
import scipy.sparse
import dask.array as da
#from igraph.drawing.text import TextDrawer
from tslearn.neighbors import KNeighborsTimeSeries
from tslearn.metrics import cdist_soft_dtw_normalized,cdist_soft_dtw
import networkx as nx
import igraph as ig
import leidenalg
import umap
from scanpy.plotting.palettes import default_20,vega_20_scanpy
from matplotlib import pyplot as plt
from matplotlib import cm
import ast
from tslearn.barycenters import softdtw_barycenter,dtw_barycenter_averaging,euclidean_barycenter
import xarray as xr

import pylab
import scipy.cluster.hierarchy as sch

import matplotlib.gridspec as gridspec
import matplotlib as mpl

import re
import goatools
import goatools.base
from goatools.base import download_go_basic_obo

from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
from goatools.semantic import semantic_similarity
from goatools.semantic import TermCounts, get_info_content

from goatools.goea.go_enrichment_ns import GOEnrichmentStudyNS
from goatools.go_enrichment import GOEnrichmentStudy

import holoviews as hv
hv.extension('bokeh')
random.seed(42)
np.random.seed(42)

warnings.filterwarnings(action='once',category=UserWarning)

import colorcet as cc

### Add Functions

In [ ]:
def highlight_regex(an_df,regex,column_label="Gene",color_all_mode="red",highlight="all",pallete=vega_20_scanpy,match_mode="regex",fontsize=18):
    ## highlight ("all","each") will either render all matches as a single color ("all") or each match type (e.g. gene name) as its own color ("each") acoording to the pallete
    ## match_mode ("regex","exact") will either perform a re.search ("regex") or exact match ("exact") for its search
    ## regex may be either (1) a regex string, (2) an exact match string or (3) a list (when in exact match mode) of exact matches to search
    
    
    highlight_df = copy.deepcopy(an_df)
    regex_title = regex
    
    if match_mode == "regex":
        re_pattern = re.compile(regex)
        regex_mask = highlight_df.obs[column_label].apply(lambda x: re.search(re_pattern,x) != None)
        regex_name_list = sorted(highlight_df.obs[regex_mask][column_label].unique().tolist())
    
    elif match_mode == "exact":
        if type(regex) == list:
            regex_mask = highlight_df.obs[column_label].apply(lambda x: any([x==regex_val for regex_val in regex]))
            regex_title = ",".join(regex_title)
        else:
            regex_mask = highlight_df.obs[column_label].apply(lambda x: x==regex)
        regex_name_list = sorted(highlight_df.obs[regex_mask][column_label].unique().tolist())
    
    if highlight == "all":
        highlight_df.obs[column_label + " | Regex: " + regex_title] = regex_mask.astype("category")
        fig = sc.pl.umap(highlight_df, title=column_label + " | Regex: " + regex_title, color=column_label + " | Regex: " + regex_title,
                     groups=[True], show=False, legend_loc='right margin', add_outline=False, size=50, return_fig=True,
                     palette = {True:color_all_mode,False:"lightgrey"});
    elif highlight == "each":
        highlight_df.obs[column_label + " | Regex: " + regex_title] = highlight_df.obs[column_label].apply(lambda x: x if x in regex_name_list else np.NaN).astype("category")
        fig = sc.pl.umap(highlight_df, title=column_label + " | Regex: " + regex_title, color=column_label + " | Regex: " + regex_title, groups=regex_name_list, show=False, legend_loc='right margin', \
                         add_outline=False, size=50, return_fig=True,palette=pallete);
    
    
    axes = fig.get_axes()
    for ax in axes:
        ax.set_title(ax.get_title(), fontsize=fontsize)
        ax.set_ylabel(ax.get_ylabel(), fontsize=fontsize)
        ax.set_xlabel(ax.get_xlabel(), fontsize=fontsize)
        
    return fig
    
def highlight_cluster(an_df,cluster_level,cluster_label,resolution_list,color="purple",fontsize=18):
    
    fig = highlight_regex(an_df,cluster_label,column_label=cluster_level,color_all_mode=color,highlight="all",match_mode="exact",fontsize=fontsize);
    
    cluster_level_index = int(cluster_level[1:])
    
    axes = fig.get_axes()
    for i,ax in enumerate(axes):
        if i%2==0:
            ax.set_title("Leiden Resolution=" + str(resolution_list[cluster_level_index]) + " | " + str(cluster_label), fontsize=fontsize)
            
    return fig

def search_go(ns2assoc, obodag, inv_gene_to_id, go_term):
    namespace_abbv = {"biological_process":"BP","molecular_function":"MF","cellular_component":"CC"}
    
    print("Searching for " + str(obodag[go_term].name))
    namespace = namespace_abbv[obodag[go_term].namespace]
    child_goterms = list(obodag[go_term].get_all_children())
    gene_list = [inv_gene_to_id[key] for key,val in ns2assoc[namespace].items() if go_term in val]
    for child_goterm in child_goterms:    
        gene_list += [inv_gene_to_id[key] for key,val in ns2assoc[namespace].items() if child_goterm in val]
    gene_list = sorted(list(set(gene_list)))
    return gene_list

def highlight_go(an_df, ns2assoc, obodag, inv_gene_to_id, go_term, go_mode = "all", color="red", fontsize=18):
    
    gene_list = search_go(ns2assoc, obodag, inv_gene_to_id, go_term)
    if go_mode == "all":
        fig = highlight_regex(an_df,gene_list,column_label="Gene",color_all_mode=color,highlight="all",match_mode="exact",fontsize=fontsize)
    elif go_mode == "each":
        fig = highlight_regex(an_df,gene_list,column_label="Gene",highlight="each",match_mode="exact",fontsize=fontsize)
        
    plt.title(go_term)
    return fig

def get_enriched_GO_terms(background_gene_list,gene_list,obodag,objanno,ns2assoc,pval=0.05,GO_type="BP"):
    
    gene_to_id = {assoc.DB_Symbol:assoc.DB_ID for assoc in objanno.associations}
    synonym_dict = {synonym:assoc.DB_ID for assoc in objanno.associations for synonym in assoc.DB_Synonym}
    gene_to_id.update(synonym_dict)

    #background gene set

    all_genes_uniprot = [gene_to_id[item] for item in background_gene_list if item in gene_to_id.keys()]
    selected_genes_uniprot = [gene_to_id[item] for item in gene_list if item in gene_to_id.keys()]
    
    print(len(all_genes_uniprot))
    print(len(selected_genes_uniprot))
    
    goeaobj = GOEnrichmentStudy(
            all_genes_uniprot, # List of mouse protein-coding genes
            ns2assoc[GO_type], # geneid/GO associations
            obodag, # Ontologies
            propagate_counts = True,
            alpha = pval, # default significance cut-off
            methods=["fdr_bh"]); # defult multipletest correction method

    goea_results_all = goeaobj.run_study(selected_genes_uniprot, prt=None)
    goea_quiet_sig = [r for r in goea_results_all if r.p_fdr_bh < pval]
    goea_quiet_enriched = [r for r in goea_quiet_sig if r.enrichment == "e"]
    return goea_quiet_enriched

def pick_exemplar(go1,go2,termcounts,obodag,info_thr,pval_factor=2.):
    
    info_1_low = get_info_content(go1.GO, termcounts)<info_thr
    info_2_low = get_info_content(go2.GO, termcounts)<info_thr
    if info_1_low and not info_2_low:
        return go2
    elif info_2_low and not info_1_low:
        return go1
    elif info_2_low and info_1_low:
        return go1
    
    pval_ratio = go1.p_fdr_bh/go2.p_fdr_bh
    
    if pval_ratio > pval_factor:
        return go2
    elif pval_ratio < (1./pval_factor):
        return go1
    
    go1_parents = list(obodag[go1.GO].get_all_parents())
    go2_parents = list(obodag[go2.GO].get_all_parents())
    
    if go2.GO in go1_parents:
        return go2
    
    elif go1.GO in go2_parents:
        return go1
    
    return go1

def get_filtered_go_terms(obodag,objanno,goea_list,sim_thr = 0.05,info_thr = 1.,GO_type="BP"):
    
    termcounts = TermCounts(obodag, objanno.get_ns2assc()[GO_type])

    go_term_list = [item.GO for item in goea_list]
    sim_arr = np.zeros((len(go_term_list),len(go_term_list)))
    for i in range(len(go_term_list)):
        for j in range(len(go_term_list)):
            sim_arr[i,j] = semantic_similarity(go_term_list[i], go_term_list[j], obodag)
    np.fill_diagonal(sim_arr, 0.)
    
    working_group_idx = 0
    grouped_terms = {}
    group_exemplars = {}
    go_term_indices = list(range(len(go_term_list)))

    while len(go_term_indices)>0:
        i = go_term_indices[0]
        most_sim_arg = np.argmax(sim_arr[i])
        sim_score = sim_arr[i,most_sim_arg]
        if sim_score > sim_thr:
            if len(grouped_terms)>0:
                in_other_group_keys = [key for key,val in grouped_terms.items() if most_sim_arg in val]
                if len(in_other_group_keys)==1:
                    other_group_idx = in_other_group_keys[0]
                    grouped_terms[other_group_idx] = grouped_terms[other_group_idx] + [i]
                    group_exemplars[other_group_idx] = pick_exemplar(group_exemplars[other_group_idx],goea_list[i],termcounts,obodag,info_thr)
                else:        
                    grouped_terms[working_group_idx] = [i,most_sim_arg]
                    group_exemplars[working_group_idx] = pick_exemplar(goea_list[i],goea_list[most_sim_arg],termcounts,obodag,info_thr)
                    working_group_idx += 1
                    go_term_indices.remove(most_sim_arg)
            else:
                grouped_terms[working_group_idx] = [i,most_sim_arg]
                group_exemplars[working_group_idx] = pick_exemplar(goea_list[i],goea_list[most_sim_arg],termcounts,obodag,info_thr)
                working_group_idx += 1
                go_term_indices.remove(most_sim_arg)
        go_term_indices.remove(i)
    
    group_exemplars = list(group_exemplars.values())
    
    return group_exemplars


def get_GO_assign_dict(selected_goea,cluster_genes_uniprot):
    all_study_items = copy.copy(cluster_genes_uniprot)
    depth_list = sorted(set([item.depth for item in selected_goea]))[::-1]
    assign_dict = {}
    for depth in depth_list:
        go_terms_at_level = [item for item in selected_goea if item.depth == depth]
        for go_term in go_terms_at_level:
            study_item_list = list(go_term.study_items)
            for study_item in study_item_list:
                if study_item in all_study_items:
                    assign_dict[study_item] = go_term.name
                    all_study_items.remove(study_item)

    for remaining_item in all_study_items:
        assign_dict[remaining_item] = "Unassigned"
        
    return assign_dict

### Reload R-compatible AnnData and Clustree Pandas Dataframe

In [ ]:
# an_df_clustree = anndata.read_h5ad("/home/de64/group/CRISPRi_Libraries/dev_notebooks/2023-02-26_lDE20_10Hour_Analysis/Z_Score_Thr_1.5_N_Neighbors_10/AnnData_nonRcompat.h5ad")

strong_effect_threshold = [1.25,1.5][1]
n_neighbors = [5,10,15][1]
resolution_list = [0.2,0.5,1.,2.,4.,10.]

rootpath = "/home/de64/group/CRISPRi_Libraries/dev_notebooks/2023-02-26_lDE20_10Hour_Analysis"
z_score_thr_dir = rootpath + "/Z_Score_Thr_" + str(strong_effect_threshold) + "_N_Neighbors_" + str(n_neighbors)
clustree_path = z_score_thr_dir + "/Clustree"
umap_path = z_score_thr_dir + "/UMAP"
compare_path = z_score_thr_dir + "/Comparison"

write_path_luis = '/home/lag36/gutiloluis/2023-05-23_DivisionEcoli'
selected_path = write_path_luis + "/Selected_Clusters"
# selected_path = z_score_thr_dir + "/Selected_Clusters"
tr.writedir(selected_path, overwrite=False)

clustree_pandas_df = pd.read_pickle(z_score_thr_dir+"/Pandas_Dataframe.pkl")
clustree_pandas_df = clustree_pandas_df[clustree_pandas_df['Large Z-score']]
an_df_clustree_r_compat = anndata.read_h5ad(z_score_thr_dir+"/AnnData.h5ad")

# Luis: Load steady state
ss_mean_cv_df = pd.read_pickle("/home/de64/scratch/de64/sync_folder/2023-01-18_lDE20_Merged_Analysis/2023-01-18_lDE20_Lineage_Cell_Cycle_Timeseries_Steady_State_MeanCV.pkl")
ss_mean_cv_df['Mean: Delta time (hr)'] = ss_mean_cv_df['Mean: Delta time (s)']/3600
ss_mean_cv_df['Variance: Delta time (hr)'] = ss_mean_cv_df['Variance: Delta time (s)']/(3600**2)
control_ss_mean_cv_df = ss_mean_cv_df[ss_mean_cv_df["Category"].isin(['OnlyPlasmid', 'NoTarget'])]

# Get ontologies
obo_fname = download_go_basic_obo()

# Get ecoli association file (ecocyc)
gaf_handle = goatools.base.http_get("http://current.geneontology.org/annotations/ecocyc.gaf.gz",fout=write_path_luis + "/ecocyc.gaf.gz")
gaf_fname = goatools.base.gunzip(write_path_luis + "/ecocyc.gaf.gz")

## Getting ontologies and other nonesense
obodag = GODag(obo_fname)
objanno = GafReader(gaf_fname)
ns2assoc = objanno.get_ns2assc()

gene_to_id = {assoc.DB_Symbol:assoc.DB_ID for assoc in objanno.associations}
inv_gene_to_id = {assoc.DB_ID:assoc.DB_Symbol for assoc in objanno.associations}
synonym_dict = {synonym:assoc.DB_ID for assoc in objanno.associations for synonym in assoc.DB_Synonym}
gene_to_id.update(synonym_dict)

In [ ]:
# Get 8h, 4h,... etc info
## remember this has already been filtered for clustering
short_expt_length_width_df = clustree_pandas_df[['Kernel Trace: Delta: Volume','Kernel Trace: Division: major_axis_length',\
                                                          'Kernel Trace: minor_axis_length',"Kernel Trace: Instantaneous Growth Rate: Volume"]]

short_expt_growth_df_t0 = short_expt_length_width_df.apply(lambda x: [column[0] for _,column in x.iteritems()])

short_expt_growth_df_t_8hr_post = short_expt_length_width_df.apply(lambda x: [column[19] for _,column in x.iteritems()])

short_expt_growth_df_t_8hr_post["Gene"] = clustree_pandas_df["Gene"]
short_expt_growth_df_t_8hr_post["TargetID"] = clustree_pandas_df["TargetID"]

### Preprocessing Time Slices 

In [ ]:
kernel_label_list = ['Kernel Trace: Delta time (hr)','Kernel Trace: Septum Displacement Length Normalized','Kernel Trace: major_axis_length',\
'Kernel Trace: minor_axis_length','Kernel Trace: mCherry mean_intensity','Kernel Trace: Instantaneous Growth Rate: Volume']
zscore_kernel_label_list = [item + ": Transformed: z score" for item in kernel_label_list]

time_slices = [11,15,19]
time_slice_labels = ["4 Hours", "6 Hours", "8 Hours"]
clustree_pandas_df['Kernel Trace: Delta time (hr): Transformed: z score'] = clustree_pandas_df['Kernel Trace: Delta time (s): Transformed: z score']

for time_slice_idx,time_slice in enumerate(time_slices):
    for kernel_label in kernel_label_list:
        zscore_kernel_label = kernel_label + ": Transformed: z score"
        
        clustree_pandas_df[time_slice_labels[time_slice_idx] + ": " + kernel_label] = \
            clustree_pandas_df[kernel_label].apply(lambda x: x[time_slice])
        
        clustree_pandas_df[time_slice_labels[time_slice_idx] + ": " + zscore_kernel_label] = \
            clustree_pandas_df[zscore_kernel_label].apply(lambda x: x[time_slice])
        
clusters_df = clustree_pandas_df.dropna(subset=["L0"])

In [ ]:
cbar_label = ""
labels = ['Mean: Delta time (hr)','Mean: Septum Displacement Length Normalized','Mean: major_axis_length',\
          'Mean: minor_axis_length','Mean: mCherry mean_intensity','Mean: Instantaneous Growth Rate: Volume']

kernel_label_list = ['Kernel Trace: Delta time (hr)','Kernel Trace: Septum Displacement Length Normalized','Kernel Trace: major_axis_length',\
'Kernel Trace: minor_axis_length','Kernel Trace: mCherry mean_intensity','Kernel Trace: Instantaneous Growth Rate: Volume']

title_list = ["Interdivision Time (hr)","Septum Displacement/Length","Length $(\mu m)$","Width $(\mu m)$",\
              "mKate2 Mean Intensity (AU)","Growth Rate (1/hr)"]

all_labels = labels

median_vals_dict = control_ss_mean_cv_df[labels].median().to_dict()
median_properties = clusters_df.groupby("L1")[labels].median()

min_vals = []
max_vals = []
norm_list = []
for label in all_labels:
    vmin,vmax = np.nanpercentile(median_properties[label],10),np.nanpercentile(median_properties[label],90)
    vcenter = median_vals_dict[label]
    min_deviation,max_deviation = (vcenter-vmin,vmax-vcenter)
    if min_deviation>max_deviation:
        vmax = vcenter + min_deviation
    else:
        vmin = vcenter - max_deviation
    min_vals.append(vmin)
    max_vals.append(vmax)

### Filamentous phenotypes Luis


In [ ]:
min_representatives = 10
cluster_level = "L1"
subcluster_level = "L2"
selected_clusters = ["2","4","5","18"]
highlighted_subclusters = ["3","12","17","25","30"] #3 for ribo

long_cluster_df = clusters_df[clusters_df[cluster_level].isin(selected_clusters)].copy()
clusters,n_rep_clusters = np.unique(long_cluster_df[subcluster_level],return_counts=True)
subclusters = clusters[n_rep_clusters>=min_representatives]
long_cluster_df = clusters_df[clusters_df[subcluster_level].isin(subclusters)].copy()
long_cluster_df.loc[:,subcluster_level] = long_cluster_df[subcluster_level].cat.remove_unused_categories().copy()

long_median_properties_list = []
for time_slice_idx,time_slice in enumerate(time_slices):
    sliced_kernel_label_list = [time_slice_labels[time_slice_idx] + ": " + kernel_label for kernel_label in kernel_label_list]
    long_median_properties = long_cluster_df.groupby(subcluster_level)[sliced_kernel_label_list].median()
    long_median_properties_list.append(long_median_properties)

In [ ]:
time_slice_idx = 2 #8 hors
long_median_properties = long_median_properties_list[time_slice_idx]

#### Do it for just the relevant clusters 

In [ ]:
unhighlighted_subclusters = [item for item in long_median_properties.index.tolist() if item not in highlighted_subclusters]
# Generate a categorical colormap using colorcet
colors = cc.glasbey_light[:len(long_median_properties.index)]
# Create a dictionary mapping each category to a color
# color_dict = dict(zip(long_median_properties.index, colors))
reorder_cluster_dict = {cluster:i for i,cluster in enumerate(long_median_properties.index.tolist())}
color_dict = {'3': '#A32CC4',
 '12': '#05acc6',
 '17': '#97ff00',
 '25': '#ffa52f',
 '30': '#ff8ec8'}

# LUIS
long_median_properties = long_median_properties[long_median_properties.index.isin(color_dict.keys())]

for subcluster in unhighlighted_subclusters:
    color_dict[subcluster] = '#808080'
categorical_colors = [color_dict[item] for item in long_median_properties.index.tolist()]
cmap = sns.color_palette(categorical_colors)
# normalize the data
norm = mpl.colors.Normalize(vmin=0, vmax=len(reorder_cluster_dict)-1)

In [ ]:
# label_names = [r'$ \tau_{C} $',r'$ L_{Sept} $',r'$ L_{C} $',r'$ W_{C} $',r'$ I_{rpsL} $',r'$ \lambda $',]

# # color_dict = {'3': '#A32CC4',
# #  '12': '#05acc6',
# #  '17': '#97ff00',
# #  '25': '#ffa52f',
# #  '30': '#ff8ec8'}

# cluster_ind = 1

# fig, axes = plt.subplots(1, len(labels)+1, figsize=(4, 1), sharey=True)
# # fig.suptitle(time_slice_labels[time_slice_idx], fontsize=16)
# plt.subplots_adjust(wspace=0)
# # Create separate single-column heatmaps for each column
# sns.heatmap(np.array([0]).reshape(-1,1),cmap=cmap,norm=norm,\
#             cbar=False, yticklabels=False, xticklabels=False, ax=axes[0],annot=False,\
#                linewidths=0,linecolor="black", annot_kws={"fontsize":14})

# cbar_list = []
# for i, label in enumerate(labels):
#     working_heatmap = sns.heatmap(long_median_properties.values[0:1,i:i+1],vmin=min_vals[i],vmax=max_vals[i],\
#                 cmap="coolwarm",center=median_vals_dict[label],\
#                cbar=False, yticklabels=False, ax=axes[i+1],annot=False,\
#                linewidths=0.5,linecolor="black")
#     axes[i+1].set_xticklabels(labels=[label_names[i]],fontsize=16)
# plt.subplots_adjust(wspace=0)

In [ ]:
cmap

In [ ]:
label_names = [r'$ \tau_{C} $',r'$ L_{Sept} $',r'$ L_{C} $',r'$ W_{C} $',r'$ I_{rpsL} $',r'$ \lambda $',]

label_names = ['Interdivision\nTime','Septum\nDisplacement','Cell Length','Cell Width','Flourescence\nIntensity','Growth Rate',]


color_dict = {'3': '#A32CC4',
 '12': '#05acc6',
 '17': '#97ff00',
 '25': '#ffa52f',
 '30': '#ff8ec8'}

fig, axes = plt.subplots(1, len(labels)+1, figsize=(4, 4), sharey=True)
# fig.suptitle(time_slice_labels[time_slice_idx], fontsize=16)
plt.subplots_adjust(wspace=0)
# Create separate single-column heatmaps for each column
# np.array([i]) for single
sns.heatmap(np.array(range(len(long_median_properties))).reshape(-1,1),cmap=cmap,norm=norm,\
            cbar=False, yticklabels=False, xticklabels=False, ax=axes[0],annot=False,\
               linewidths=0,linecolor="black", annot_kws={"fontsize":2})

cbar_list = []
for i, label in enumerate(labels):
    working_heatmap = sns.heatmap(long_median_properties.values[:,i:i+1],vmin=min_vals[i],vmax=max_vals[i],\
                cmap="coolwarm",center=median_vals_dict[label],\
               cbar=False, yticklabels=False, ax=axes[i+1],annot=False,\
               linewidths=0.5,linecolor="black")
    axes[i+1].set_xticklabels(labels=[label_names[i]],fontsize=12, rotation = 90)
plt.subplots_adjust(wspace=0)
# plt.savefig("/home/lag36/gutiloluis/2023-05-23_DivisionEcoli/Matrix_Div_Clusters.png",
#             dpi=300,transparent=False, bbox_inches='tight')

### Only 4 variables

In [ ]:
plt.style.available

In [ ]:
plt.style.use('seaborn-v0_8-white')

In [ ]:
# plt.style.use('dark_background')
inds_to_keep = [2,1,3,5]
min_vals_short = list(np.array(min_vals)[inds_to_keep])
max_vals_short = list(np.array(max_vals)[inds_to_keep])

long_median_properties_short = long_median_properties.iloc[:,inds_to_keep]
labels_short = list(np.array(labels)[inds_to_keep])
label_names_short = list(np.array(label_names)[inds_to_keep])

color_dict = {'3': '#A32CC4',
 '12': '#05acc6',
 '17': '#97ff00',
 '25': '#ffa52f',
 '30': '#ff8ec8'}

fig, axes = plt.subplots(1, len(labels_short)+1, figsize=(4, 4), sharey=True)

plt.subplots_adjust(wspace=0)
# Create separate single-column heatmaps for each column
# np.array([i]) for single
sns.heatmap(np.array(range(len(long_median_properties_short))).reshape(-1,1),cmap=cmap,norm=norm,\
            cbar=False, yticklabels=False, xticklabels=False, ax=axes[0],annot=False,\
               linewidths=0,linecolor="black", annot_kws={"fontsize":2})

cbar_list = []
for i, label in enumerate(labels_short):
    working_heatmap = sns.heatmap(long_median_properties_short.values[:,i:i+1],vmin=min_vals_short[i],vmax=max_vals_short[i],\
                cmap="coolwarm",center=median_vals_dict[label],\
               cbar=False, yticklabels=False, ax=axes[i+1],annot=False,\
               linewidths=0.5,linecolor="black")
    axes[i+1].set_xticklabels(labels=[label_names_short[i]],fontsize=15, rotation = 90)
plt.subplots_adjust(wspace=0)
# plt.savefig("/home/lag36/gutiloluis/2023-05-23_DivisionEcoli/Matrix_Div_Clusters.png",
#             dpi=300,transparent=False, bbox_inches='tight')

## Matrix Over time

#### nrdA, B, pyrG
Use nucleoids and timescale as an argument for division defect and not a generic replication one

In [ ]:
def compute_median_and_mad_trace(clust_lev='L2', clust_num='3', column='Kernel Trace: major_axis_length'):
    a = clustree_pandas_df.loc[clustree_pandas_df[clust_lev] == clust_num, column]
    list_median = np.zeros(20)
    list_mad = np.zeros(20)
    for i in range(len(list_median)):
        list_median[i] = a.str[i].median()
        list_mad[i] = a.str[i].mad()
    return list_median, list_mad

color_dict = {'3': '#A32CC4',
 '12': '#05acc6',
 '17': '#97ff00',
 '25': '#ffa52f',
 '30': '#ff8ec8'}
def plot_med_traces(color_dict):
    # plt.figure(figsize=(4,4))
    for i, clust_num in enumerate(color_dict):
        list_median, list_mad = compute_median_and_mad_trace('L2', clust_num, 'Kernel Trace: major_axis_length')
        # print(len(list_median))
        x = np.arange(-2,8,0.5)
        
        plt.plot(x,list_median, c=color_dict[clust_num])
        plt.xlabel('Time post-induction (h)', fontsize = 12)
        plt.ylabel('Mean Cell Length ($\mu$m)', fontsize=12)
        plt.xlim([-2,8])

In [ ]:
plt.figure(figsize=(4,4))

clust_lev='L2'
clust_num='12'
column = 'Kernel Trace: major_axis_length'
gene = 'nrdA'
x = np.arange(-2,8,0.5)
a = clustree_pandas_df.loc[(clustree_pandas_df[clust_lev] == clust_num) & 
                       (clustree_pandas_df['Gene'] == gene), column]
list_median = np.zeros(20)
list_mad = np.zeros(20)
for i in range(len(list_median)):
    list_median[i] = a.str[i].median()
    list_mad[i] = a.str[i].mad()
plt.plot(x, list_median, label = gene)


clust_lev='L2'
clust_num='12'
column = 'Kernel Trace: major_axis_length'
gene = 'nrdB'
a = clustree_pandas_df.loc[(clustree_pandas_df[clust_lev] == clust_num) & 
                       (clustree_pandas_df['Gene'] == gene), column]
list_median = np.zeros(20)
list_mad = np.zeros(20)
for i in range(len(list_median)):
    list_median[i] = a.str[i].median()
    list_mad[i] = a.str[i].mad()
plt.plot(x,list_median, label = gene)


clust_lev='L2'
clust_num='12'
column = 'Kernel Trace: major_axis_length'
gene = 'pyrG'
a = clustree_pandas_df.loc[(clustree_pandas_df[clust_lev] == clust_num) & 
                       (clustree_pandas_df['Gene'] == gene), column]
list_median = np.zeros(20)
list_mad = np.zeros(20)
for i in range(len(list_median)):
    list_median[i] = a.str[i].median()
    list_mad[i] = a.str[i].mad()
plt.plot(x,list_median, label = gene)


clust_lev='L2'
clust_num='12'
column = 'Kernel Trace: major_axis_length'
gene = 'pgsA'
a = clustree_pandas_df.loc[(clustree_pandas_df[clust_lev] == clust_num) & 
                       (clustree_pandas_df['Gene'] == gene), column]
list_median = np.zeros(20)
list_mad = np.zeros(20)
for i in range(len(list_median)):
    list_median[i] = a.str[i].median()
    list_mad[i] = a.str[i].mad()
plt.plot(x,list_median, label = gene)


clust_lev='L2'
clust_num='12'
column = 'Kernel Trace: major_axis_length'
gene = 'cdsA'
a = clustree_pandas_df.loc[(clustree_pandas_df[clust_lev] == clust_num) & 
                       (clustree_pandas_df['Gene'] == gene), column]
list_median = np.zeros(20)
list_mad = np.zeros(20)
for i in range(len(list_median)):
    list_median[i] = a.str[i].median()
    list_mad[i] = a.str[i].mad()
plt.plot(x,list_median, label = gene)




color_dict = {'3': '#A32CC4',
 '12': '#05acc6',
 '17': '#97ff00',
 '25': '#ffa52f',
 '30': '#ff8ec8'}

plot_med_traces(color_dict)

plt.legend()
# list_median, list_mad = compute_median_and_mad_trace('L2', clust_num, 'Kernel Trace: major_axis_length')

In [ ]:
plt.style.use('dark_background')
plt.figure(figsize=(3,3))
plot_med_traces(color_dict)
plt.savefig("/home/lag36/gutiloluis/2023-05-23_DivisionEcoli/Traces_Div_Clusters.png",
            dpi=300,transparent=False, bbox_inches='tight')

In [ ]:
plt.errorbar(x=np.arange(len(list_median)), 
             y=list_median, yerr=list_mad)

## Only L2 with relevant clusters

### All gray

In [ ]:
# All gray

color_dict = {'3': '#A32CC4',
 '12': '#05acc6',
 '17': '#97ff00',
 '25': '#ffa52f',
 '30': '#ff8ec8'}
gray_but_div = 32*['#bebebe']

leiden_labels = ['L2']
# leiden_titles = ["Leiden Resolution=" + str(res) for i,res in enumerate(resolution_list)][:3]
fig = sc.pl.umap(an_df_clustree_r_compat, color=leiden_labels, show=False, title = '',
           legend_loc=None, edges=False, add_outline=False, size=100, return_fig=True,palette=gray_but_div,ncols=2)
axes = fig.get_axes()
for ax in axes:
#     ax.set_title(ax.get_title(), fontsize=18)
#     ax.set_ylabel(ax.get_ylabel(), fontsize=18)
#     ax.set_xlabel(ax.get_xlabel(), fontsize=18)

    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.axis('off')

plt.savefig("/home/lag36/gutiloluis/2023-05-23_DivisionEcoli/UMAP_All_Gray.png",dpi=300,transparent=True)

### All colors

In [ ]:
cmap_l2 = vega_20_scanpy[:] + vega_20_scanpy[:12]
color_dict = {'3': '#A32CC4',
 '12': '#05acc6',
 '17': '#97ff00',
 '25': '#ffa52f',
 '30': '#ff8ec8'}
for div_c in color_dict:
    cmap_l2[int(div_c)] = color_dict[div_c]
    
# Changing colors for proximity
cmap_l2[4] = cmap_l2[5]

leiden_labels = ['L2']
# leiden_titles = ["Leiden Resolution=" + str(res) for i,res in enumerate(resolution_list)][:3]
fig = sc.pl.umap(an_df_clustree_r_compat, color=leiden_labels, show=False, title='',
           legend_loc=None, edges=False, add_outline=False, size=100, return_fig=True,palette=cmap_l2,ncols=2)
axes = fig.get_axes()
for ax in axes:
    # ax.set_title(ax.get_title(), fontsize=18)
    # ax.set_ylabel(ax.get_ylabel(), fontsize=18)
    # ax.set_xlabel(ax.get_xlabel(), fontsize=18)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.axis('off')
plt.savefig("/home/lag36/gutiloluis/2023-05-23_DivisionEcoli/UMAP_All_Clusters_L2.png",dpi=300,transparent=True)

### Only clusters of interest

In [ ]:
color_dict = {'3': '#A32CC4',
 '12': '#05acc6',
 '17': '#97ff00',
 '25': '#ffa52f',
 '30': '#ff8ec8'}
gray_but_div = 32*['#bebebe']
for div_c in color_dict:
    gray_but_div[int(div_c)] = color_dict[div_c]

leiden_labels = ['L2']
# leiden_titles = ["Leiden Resolution=" + str(res) for i,res in enumerate(resolution_list)][:3]
fig = sc.pl.umap(an_df_clustree_r_compat, color=leiden_labels, show=False, title = '',
           legend_loc=None, edges=False, add_outline=False, size=100, return_fig=True,palette=gray_but_div,ncols=2)
axes = fig.get_axes()
for ax in axes:
#     ax.set_title(ax.get_title(), fontsize=18)
#     ax.set_ylabel(ax.get_ylabel(), fontsize=18)
#     ax.set_xlabel(ax.get_xlabel(), fontsize=18)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.axis('off')
plt.savefig("/home/lag36/gutiloluis/2023-05-23_DivisionEcoli/UMAP_Div_Clusters.png",dpi=300,transparent=True)

In [ ]:
color_dict = {#'3': '#A32CC4',
 '12': '#05acc6',
 '17': '#97ff00',
 '25': '#ffa52f',
}#'30': '#ff8ec8'}
gray_but_div = 32*['#bebebe']
for div_c in color_dict:
    gray_but_div[int(div_c)] = color_dict[div_c]

leiden_labels = ['L2']
# leiden_titles = ["Leiden Resolution=" + str(res) for i,res in enumerate(resolution_list)][:3]
fig = sc.pl.umap(an_df_clustree_r_compat, color=leiden_labels, show=False, title = '',
           legend_loc=None, edges=False, add_outline=False, size=100, return_fig=True,palette=gray_but_div,ncols=2)
axes = fig.get_axes()
for ax in axes:
#     ax.set_title(ax.get_title(), fontsize=18)
#     ax.set_ylabel(ax.get_ylabel(), fontsize=18)
#     ax.set_xlabel(ax.get_xlabel(), fontsize=18)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    ax.axis('off')
plt.savefig("/home/lag36/gutiloluis/2023-05-23_DivisionEcoli/UMAP_Div_Clusters.png",dpi=300,transparent=True)

### Clusters with Length Cmap

In [ ]:
all_labels = ['Mean_major_axis_length']# , 'Mean_Septum_Displacement_Length_Normalized', 'Mean_mCherry_mean_intensity']
title_list = ['Mean Cell Length ($\mu$m)']

fig = sc.pl.umap(an_df_clustree_r_compat, color=all_labels, show=False,
                 title=title_list,legend_loc='on data', add_outline=False, size=50,
                 return_fig=True,cmap='RdBu_r',wspace=0.25,ncols=2,
                vmin=min_vals[2], vmax=max_vals[2], vcenter=median_vals_dict['Mean: major_axis_length'])

axes = fig.get_axes()
for ax in axes:
#     ax.set_title(ax.get_title(), fontsize=18)
#     ax.set_ylabel(ax.get_ylabel(), fontsize=18)
#     ax.set_xlabel(ax.get_xlabel(), fontsize=18)
    # ax.get_xaxis().set_visible(False)
    # ax.get_yaxis().set_visible(False)
    # ax.axis('off')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.set(xlabel=None, ylabel=None)
    # ax.set_title(title_list[0], color='white')
# cb = fig.colorbar(axes) 
# plt.colorbar(fig)
# cb.remove()
# plt.savefig("/home/lag36/gutiloluis/2023-05-23_DivisionEcoli/UMAP_Length_Cmap.png",dpi=300,transparent=True)

#### Labels with several levels of resolution

In [ ]:
leiden_labels = ["L" + str(i) for i,res in enumerate(resolution_list)][:3]
leiden_titles = ["Leiden Resolution=" + str(res) for i,res in enumerate(resolution_list)][:3]
fig = sc.pl.umap(an_df_clustree_r_compat, color=leiden_labels, title=leiden_titles, show=False,
           legend_loc='on data', edges=False, add_outline=False, size=100, return_fig=True,palette=vega_20_scanpy,ncols=2)
axes = fig.get_axes()
for ax in axes:
    ax.set_title(ax.get_title(), fontsize=18)
    ax.set_ylabel(ax.get_ylabel(), fontsize=18)
    ax.set_xlabel(ax.get_xlabel(), fontsize=18)

## Gene names

In [ ]:
an_df_clustree = an_df_clustree_r_compat
an_obs = an_df_clustree.obs
len(list(an_obs[an_obs['L2'] == '12']['Gene'].unique()))

In [ ]:
n_gene_counts_min = 1
an_obs_div = an_obs[an_obs['L2'].isin(color_dict.keys())]
gene_counts = an_obs_div.groupby("Gene").size()
genes_filtered = gene_counts[gene_counts>=n_gene_counts_min].index.tolist()
an_obs_div = an_obs_div[an_obs_div["Gene"].isin(genes_filtered)]

In [ ]:
genes_in_cluster = list(an_obs_div[an_obs_div['L2'] == '3']['Gene'].unique())
large_ribo_subunit = []
small_ribo_subunit = []
rpm = []
rest = []
for gene in genes_in_cluster:
    if 'rpl' in gene:
        large_ribo_subunit.append(gene)
    elif 'rps' in gene:
        small_ribo_subunit.append(gene)
    elif 'rpm' in gene:
        rpm.append(gene)
    else:
        rest.append(gene)
    
print('rpl: ', end='')
for gene in sorted(large_ribo_subunit):
    print(gene[3], end=', ')
print()

print('rps: ', end='')
for gene in sorted(small_ribo_subunit):
    print(gene[3], end=', ')
print()

print('rpm: ', end='')
for gene in sorted(rpm):
    print(gene[3], end=', ')
print()

for gene in sorted(rest):
    print(gene)
print()

In [ ]:
print(1)

In [ ]:
genes_in_cluster = list(an_obs_div[an_obs_div['L2'] == '3']['Gene'].unique())
# print(sorted(genes_in_cluster))
for gene in sorted(genes_in_cluster):
    print(gene, end=', ')

In [ ]:
genes_in_cluster = list(an_obs_div[an_obs_div['L2'] == '12']['Gene'].unique())
print(sorted(genes_in_cluster))
for gene in sorted(genes_in_cluster):
    print(gene, end=', ')

In [ ]:
genes_in_cluster = list(an_obs_div[an_obs_div['L2'] == '25']['Gene'].unique())
# print(sorted(genes_in_cluster))
for gene in genes_in_cluster:
    print(gene, end=', ')

In [ ]:
genes_in_cluster = list(an_obs_div[an_obs_div['L2'] == '30']['Gene'].unique())
print(sorted(genes_in_cluster))

In [ ]:
# Generate kymos and do it from nucleoid kymos

In [ ]:
,

### Kymos

In [ ]:
# TODO Write cleanly
# I


In [ ]:
print(1)

In [ ]:
import h5py

In [ ]:
df26 = pd.read_pickle("/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-25_sgRNA_Timeseries_df.pkl")
df28 = pd.read_pickle("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_sgRNA_Timeseries_df.pkl")

In [ ]:
clustree_pandas_df.loc[clustree_pandas_df['L2'] == '25', ['Gene', 'N Mismatch', 'N Observations']]['Gene'].unique()

In [ ]:
clustree_pandas_df.loc[(clustree_pandas_df['L2'] == '25') & (clustree_pandas_df['Gene'] == 'mukE')]

In [ ]:
clustree_pandas_df.loc[clustree_pandas_df['L2'] == '12', ['Gene', 'N Mismatch', 'N Observations']]

In [ ]:
chosen_gene = 'secE'
df26.loc[df26['Gene'] == chosen_gene,  ['Gene', 'N Mismatch', 'N Observations']]

In [ ]:
from IPython.display import Video
oDE_id = 1396
Video("/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/MP4_Library/Merged/"+str(oDE_id)+".mp4", embed=True)

In [ ]:
trench_to_lookup_df26 = pd.read_pickle('/home/lag36/gutiloluis/2023-03-23_lDE26-Run1/lDE26_trench_to_lookup_df.pkl')
trench_to_lookup_df28 = pd.read_pickle('/home/lag36/gutiloluis/2023-03-23_lDE26-Run1/lDE28_trench_to_lookup_df.pkl')

In [ ]:
trench_to_lookup_df26[trench_to_lookup_df26['oDEPool7_id'] == 21472]

In [ ]:
import h5py

In [ ]:
kymo_path_26 = '/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/Growth_Division/kymograph/kymograph_'
kymo_path_28 = '/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/Growth_Division/kymograph/kymograph_'

In [ ]:
file_idx = 6
idx = 91
channels = ['YFP', 'mCherry']
# t_inds = np.arange(0,61,12)
t_inds = np.arange(0,61,12)
print(t_inds)
with h5py.File(kymo_path_26 + str(file_idx) + ".hdf5", "r") as infile:  
    for ch in channels:
        img = infile[ch][idx][t_inds[0]]
        if len(t_inds) > 1:
            for t_ind in t_inds[1:]:
                img_ind = infile[ch][idx][t_ind]
                img = np.concatenate((img, img_ind), axis = 1)

        fig, ax = plt.subplots();
        ax.imshow(img, cmap='gray');
        # ax.set_axis_off()
        plt.yticks(ticks=[],labels=[])
        plt.xticks(ticks=[],labels=[])
        plt.savefig("/home/lag36/gutiloluis/2023-05-23_DivisionEcoli/Kymos_Div_Genes_"+chosen_gene+'_'
                    +str(file_idx)+'_'+str(idx)+'_'+ str(ch) + ".png",dpi=500, transparent=True)

In [ ]:
file_idx = 372	
idx = 14
channels = ['YFP', 'mCherry']
# t_inds = np.arange(0,61,12)
t_inds = np.arange(0,61,12)
print(t_inds)
with h5py.File(kymo_path_26 + str(file_idx) + ".hdf5", "r") as infile:  
    for ch in channels:
        img = infile[ch][idx][t_inds[0]]
        if len(t_inds) > 1:
            for t_ind in t_inds[1:]:
                img_ind = infile[ch][idx][t_ind]
                img = np.concatenate((img, img_ind), axis = 1)

        fig, ax = plt.subplots();
        ax.imshow(img, cmap='gray');
        plt.yticks(ticks=[],labels=[])
        plt.xticks(ticks=[],labels=[])
        # plt.savefig("/home/lag36/gutiloluis/2023-03-23_lDE26-Run1/Powerpoint_rne_Kymo_1_" + str(ch) + ".png",dpi=500)

In [ ]:
trench_to_lookup_df28[trench_to_lookup_df28['oDEPool7_id'] == 3498]

In [ ]:
file_idx = 5
idx = 60
channels = ['YFP', 'mCherry']
t_inds = np.arange(0,61,12)
print(t_inds)
with h5py.File(kymo_path_28 + str(file_idx) + ".hdf5", "r") as infile:  
    for ch in channels:
        img = infile[ch][idx][t_inds[0]]
        if len(t_inds) > 1:
            for t_ind in t_inds[1:]:
                img_ind = infile[ch][idx][t_ind]
                img = np.concatenate((img, img_ind), axis = 1)

        fig, ax = plt.subplots();
        ax.imshow(img, cmap='gray');
        plt.yticks(ticks=[],labels=[])
        plt.xticks(ticks=[],labels=[])

## Nucleoids!

In [ ]:
df26 = pd.read_pickle("/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/2023-04-25_sgRNA_Timeseries_df.pkl")
df28 = pd.read_pickle("/home/lag36/scratch/lag36/2023-03-25_lDE28_Run_1/2023-04-24_lDE28_sgRNA_Timeseries_df.pkl")

In [ ]:
col_x='Kernel Trace: Instantaneous Growth Rate: Volume'
col_y='Kernel Trace: major_axis_length'

In [ ]:
cl3_ids = long_cluster_df[long_cluster_df['L2']=='3'].index

In [ ]:
def get_subset_df(df, GO_id="GO:0006633"):
    gene_names = search_go(ns2assoc, obodag, inv_gene_to_id, GO_id)
    return gene_names, df[df["Gene"].isin(gene_names)]

In [ ]:
def get_two_timepoints_two_vars(df, col_x, col_y, ti, tf):

    xf = df[col_x].str[tf]
    yf = df[col_y].str[tf]
    if ti == None:
        return xf, yf
    else:
        xi = df[col_x].str[ti]
        yi = df[col_y].str[ti]
   
        return xf, yf, xi, yi

In [ ]:
trna_synthetases_GO = "GO:0006418"
fatty_acid_biosyn_GO = "GO:0006633"
large_ribo_GO = "GO:0022625"
small_ribo_GO = "GO:0022627"
dna_rep_GO = "GO:0006260"
rrna_processing_GO = "GO:0006364"
ribosome_biogen_GO = "GO:0042254"

trna_synthetases = search_go(ns2assoc, obodag, inv_gene_to_id, trna_synthetases_GO)
large_ribo = search_go(ns2assoc, obodag, inv_gene_to_id, large_ribo_GO)
small_ribo = search_go(ns2assoc, obodag, inv_gene_to_id, small_ribo_GO)

In [ ]:
# df_sub[(df_sub['Kernel Trace: Instantaneous Growth Rate: Volume'].apply(lambda x: x[-1])<0.6)&\
# (df_sub['Kernel Trace: Total Nucleoid Area per Cell Area'].apply(lambda x: x[-1])<0.51)]["Gene"]

In [ ]:
color_dict = {'3': '#A32CC4', #ribo
 '12': '#05acc6', #div
 '17': '#97ff00', #rep
 '25': '#ffa52f', # sep
 '30': '#ff8ec8'} # thyA

In [ ]:
list(df26.columns)

In [ ]:
col_x='Kernel Trace: Instantaneous Growth Rate: Volume'
col_x='Kernel Trace: major_axis_length'
col_y='Kernel Trace: Total Nucleoid Area per Cell Area'
# col_y='Kernel Trace: Total Nucleoid Length per Cell Length'
# col_y='Kernel Trace: Total Number of Nucleoids per Cell Length'

xf, yf, xi, yi = get_two_timepoints_two_vars(df=df26,
                                             col_x=col_x,
                                             col_y=col_y,
                                             ti=0, tf=19)
fig = plt.figure()
plt.scatter(xf, yf, s=2,color="gray")

for i, cluster_num in enumerate(color_dict):
    cluster_oDEids = long_cluster_df[long_cluster_df['L2']==cluster_num].index
    df_sub = df26[df26.index.isin(cluster_oDEids)]
    xf, yf  = get_two_timepoints_two_vars(df=df_sub,
                                          col_x=col_x,
                                          col_y=col_y,
                                          ti=None, tf=19)
    plt.scatter(xf, yf, s=6, label="Large Subunit",color=color_dict[cluster_num])

In [ ]:
cluster_num = '17'

In [ ]:
df_sub[col_y].str[-1]

In [ ]:
cluster_oDEids

In [ ]:
cluster_oDEids = long_cluster_df[long_cluster_df['L2']==cluster_num].index
df_sub = df26[df26.index.isin(cluster_oDEids)]

In [ ]:
a = df_sub[df_sub[col_y].str[-1] > 0.6][['Gene', col_y]]
a

In [ ]:
from IPython.display import Video
oDE_id = 20000
Video("/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/MP4_Library/Merged/"+str(oDE_id)+".mp4", embed=True)

In [ ]:
a[col_y].str[-1]

In [ ]:
col_x='Kernel Trace: Instantaneous Growth Rate: Volume'
col_x='Kernel Trace: major_axis_length'
col_x='Kernel Trace: Number of Nucleoids'
# col_y='Kernel Trace: Total Nucleoid Area per Cell Area'
col_y='Kernel Trace: Total Nucleoid Length per Cell Length'
# col_y='Kernel Trace: Total Number of Nucleoids per Cell Length'

xf, yf, xi, yi = get_two_timepoints_two_vars(df=df26,
                                             col_x=col_x,
                                             col_y=col_y,
                                             ti=0, tf=19)
fig = plt.figure()

plt.scatter(xf, yf, s=2,color="gray")
plt.scatter(xi, yi, s=2,color='white')
for i, cluster_num in enumerate(color_dict):
    cluster_oDEids = long_cluster_df[long_cluster_df['L2']==cluster_num].index
    df_sub = df26[df26.index.isin(cluster_oDEids)]
    xf, yf  = get_two_timepoints_two_vars(df=df_sub,
                                          col_x=col_x,
                                          col_y=col_y,
                                          ti=None, tf=19)
    plt.scatter(xf, yf, s=10, label="Large Subunit",color=color_dict[cluster_num])

plt.hlines(0.85, 0.9, 2.7, linestyles='dashed')
plt.hlines(0.72, 0.9, 2.7, linestyles='dashed')
plt.vlines(1.6, 0.4, 1,linestyles='dashed')
plt.xlim([0.9,2.7])
plt.ylim([0.4,1])
plt.xlabel('Mean Number of Nucleoids per Cell', fontsize=14)
plt.ylabel('Mean Nucleoid Length per Cell Length', fontsize=14)
# plt.savefig("/home/lag36/gutiloluis/2023-05-23_DivisionEcoli/Nuc_L_vs_Nuc_N.png",
#             dpi=300,transparent=False, bbox_inches='tight')

In [ ]:
sorted(df26.loc[(df26['Kernel Trace: Total Nucleoid Length per Cell Length'].str[-1] <= 0.85) &
         (df26['Kernel Trace: Total Nucleoid Length per Cell Length'].str[-1] >= 0.72) &
         (df26['Kernel Trace: Number of Nucleoids'].str[-1] >= 1.6), 'Gene'].unique())

In [ ]:
df26[df26['Category'] == 'NoTarget']

In [ ]:
col_x='Kernel Trace: Instantaneous Growth Rate: Volume'
col_x='Kernel Trace: major_axis_length'
# col_x='Kernel Trace: Number of Nucleoids'
# col_y='Kernel Trace: Total Nucleoid Area per Cell Area'
col_y='Kernel Trace: Total Nucleoid Length per Cell Length'
# col_y='Kernel Trace: Total Number of Nucleoids per Cell Length'

xf, yf, xi, yi = get_two_timepoints_two_vars(df=df26,
                                             col_x=col_x,
                                             col_y=col_y,
                                             ti=0, tf=19)
fig = plt.figure()
plt.scatter(xf, yf, s=8,color="black")

plt.scatter(xi, yi, s=8, color='blue', label = 'Pre-induction')

for i, cluster_num in enumerate(color_dict):
    cluster_oDEids = long_cluster_df[long_cluster_df['L2']==cluster_num].index
    df_sub = df26[df26.index.isin(cluster_oDEids)]
    xf, yf  = get_two_timepoints_two_vars(df=df_sub,
                                          col_x=col_x,
                                          col_y=col_y,
                                          ti=None, tf=19)
    plt.scatter(xf, yf, s=18,color=color_dict[cluster_num])
    
xf, yf = get_two_timepoints_two_vars(df=df26[df26['Category'] == 'NoTarget'],
                                     col_x=col_x,
                                      col_y=col_y,
                                      ti=None, tf=19)

plt.scatter(xf, yf, s=8,color='red', label = 'No target controls')
    
plt.xlim([1.5,16])
plt.xlabel('Cell Length', fontsize=14)
plt.ylabel('Nucleoid Length per Cell Length', fontsize=14)
plt.legend()
plt.savefig("/home/lag36/gutiloluis/2023-05-23_DivisionEcoli/NucLperCellL-vs-Length_White" + ".png",dpi=500, transparent=True)

In [ ]:
print(1)

In [ ]:
list(df26.columns)

In [ ]:
col_x='Kernel Trace: Instantaneous Growth Rate: Volume'
col_x='Kernel Trace: major_axis_length'
# col_x='Kernel Trace: Number of Nucleoids'
# col_y='Kernel Trace: Total Nucleoid Area per Cell Area'
col_y='Kernel Trace: Total Nucleoid Length per Cell Length'
# col_y='Kernel Trace: Total Number of Nucleoids per Cell Length'

xf, yf, xi, yi = get_two_timepoints_two_vars(df=df26,
                                             col_x=col_x,
                                             col_y=col_y,
                                             ti=0, tf=19)
fig = plt.figure()
plt.scatter(xf, yf, s=6,color="black")

plt.scatter(xi, yi, s=6, color='blue', label = 'Pre-induction')

for i, cluster_num in enumerate(color_dict):
    cluster_oDEids = long_cluster_df[long_cluster_df['L2']==cluster_num].index
    df_sub = df26[df26.index.isin(cluster_oDEids)]
    xf, yf  = get_two_timepoints_two_vars(df=df_sub,
                                          col_x=col_x,
                                          col_y=col_y,
                                          ti=None, tf=19)
    plt.scatter(xf, yf, s=6,color=color_dict[cluster_num])
    
xf, yf = get_two_timepoints_two_vars(df=df26[df26['Category'] == 'NoTarget'],
                                     col_x=col_x,
                                      col_y=col_y,
                                      ti=None, tf=19)

plt.scatter(xf, yf, s=8,color='red', label = 'No target controls')
    
plt.xlabel('Cell Length', fontsize=14)
plt.ylabel('Nucleoid Length per Cell Length', fontsize=14)
plt.legend()
# plt.savefig("/home/lag36/gutiloluis/2023-05-23_DivisionEcoli/Kymos_Div_Genes_"+chosen_gene+'_'
#                     +str(file_idx)+'_'+str(idx)+'_'+ str(ch) + ".png",dpi=500, transparent=True)

In [ ]:
df26[df26[

In [ ]:
col_x='Kernel Trace: Instantaneous Growth Rate: Volume'
# col_x='Kernel Trace: major_axis_length'
# col_x='Kernel Trace: Number of Nucleoids'
# col_y='Kernel Trace: Total Nucleoid Area per Cell Area'
col_y='Kernel Trace: Total Nucleoid Length per Cell Length'
# col_y='Kernel Trace: Total Number of Nucleoids per Cell Length'

xf, yf, xi, yi = get_two_timepoints_two_vars(df=df26,
                                             col_x=col_x,
                                             col_y=col_y,
                                             ti=0, tf=19)
fig = plt.figure()
plt.scatter(xf, yf, s=2,color="gray")

for i, cluster_num in enumerate(color_dict):
    cluster_oDEids = long_cluster_df[long_cluster_df['L2']==cluster_num].index
    df_sub = df26[df26.index.isin(cluster_oDEids)]
    xf, yf  = get_two_timepoints_two_vars(df=df_sub,
                                          col_x=col_x,
                                          col_y=col_y,
                                          ti=None, tf=19)
    plt.scatter(xf, yf, s=6, label="Large Subunit",color=color_dict[cluster_num])

In [ ]:
# Make kymographs

In [ ]:
df26[df26['Gene'] == 'cdsA']['N Mismatch']

In [ ]:
from IPython.display import Video
oDE_id = 7364
Video("/home/lag36/scratch/lag36/2023-03-11_lDE26_Run_1/MP4_Library/Merged/"+str(oDE_id)+".mp4", embed=True)

In [ ]:
xf, yf, xi, yi = get_two_timepoints_two_vars(df=df26,
                                             col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
                                             col_y='Kernel Trace: Total Nucleoid Area per Cell Area',
                                             ti=0, tf=19)

plt.scatter(xf, yf, s=2,color="black")
# plt.scatter(xi, yi, s=1)

# xf, yf, xi, yi = get_two_timepoints_two_vars(df=df28,
#                                              col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
#                                              col_y='Kernel Trace: Total Nucleoid Area per Cell Area',
#                                              ti=0, tf=19)

# plt.scatter(xf, yf, s=3)
# plt.scatter(xi, yi, s=3)

_, df_sub = get_subset_df(df=df26, GO_id=large_ribo_GO)
xf, yf  = get_two_timepoints_two_vars(df=df_sub,
                                      col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
                                      col_y='Kernel Trace: Total Nucleoid Area per Cell Area',
                                      ti=None, tf=19)
plt.scatter(xf, yf, s=6, label="Large Subunit",color="tab:blue")

_, df_sub = get_subset_df(df=df26, GO_id=small_ribo_GO)
xf, yf  = get_two_timepoints_two_vars(df=df_sub,
                                      col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
                                      col_y='Kernel Trace: Total Nucleoid Area per Cell Area',
                                      ti=None, tf=19)

plt.scatter(xf, yf, s=6, label="Small Subunit",color="salmon")

_, df_sub_1 = get_subset_df(df=df26, GO_id=ribosome_biogen_GO)
_, df_sub_2 = get_subset_df(df=df26, GO_id=rrna_processing_GO)
all_sgrnas = list(set(df_sub_1.index.tolist())|set(df_sub_2.index.tolist()))
df_sub = df26.loc[all_sgrnas]
# filter out off-target
df_sub = df_sub[df_sub["Gene"]!="rimM"]
df_sub = df_sub[df_sub["Gene"]!="rpsJ"]

xf, yf  = get_two_timepoints_two_vars(df=df_sub,
                                      col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
                                      col_y='Kernel Trace: Total Nucleoid Area per Cell Area',
                                      ti=None, tf=19)

plt.scatter(xf, yf, s=20, label="Ribosome Maturation",color="tab:olive")
plt.ylabel(r"$ A_{Nuc}/A_{Cell} $",fontsize=18)
plt.xlabel("Growth Rate (1/hr)",fontsize=18)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

plt.legend(fontsize=14)

# xf, yf  = get_two_timepoints_two_vars(df=df26[df26["Gene"]=="rne"],
#                                       col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
#                                       col_y='Kernel Trace: Total Nucleoid Area per Cell Area',
#                                       ti=None, tf=19)

# plt.scatter(xf, yf, s=5)

# _, df_sub = get_subset_df(df=df28, GO_id=trna_synthetases_GO)
# xf, yf  = get_two_timepoints_two_vars(df=df_sub,
#                                       col_x='Kernel Trace: Instantaneous Growth Rate: Volume',
#                                       col_y='Kernel Trace: major_axis_length',
#                                       ti=None, tf=19)
# plt.scatter(xf, yf, s=5)
plt.tight_layout()
plt.savefig("/home/lag36/gutiloluis/2023-03-23_lDE26-Run1/Powerpoint_nuc_compact.png",dpi=500)

### UMAP: Parameter Overlay

# Shading by a variable

In [ ]:
cbar_label = ""
labels = ['Mean: Delta time (hr)','Mean: Septum Displacement Length Normalized','Mean: major_axis_length',\
          'Mean: minor_axis_length','Mean: mCherry mean_intensity','Mean: Instantaneous Growth Rate: Volume']

title_list = ["Interdivision Time (hr)","Septum Displacement/Length","Length $(\mu m)$","Width $(\mu m)$",\
              "mKate2 Mean Intensity (AU)","Growth Rate (1/hr)"]#,"Pico488 Mean Intensity (AU)"]


### UMAP: Cluster Timeseries

In [ ]:
def plot_cluster_timeseries(df, cluster_label, feature_labels, displayed_labels, feature_range_list, agg_fn=np.mean, x_ticks = [0,10,20], x_ticklabels=[0,5,10], cluster_subset=None, figsize=(10,10), wspace=0., hspace=0., fontsize=14, linewidth=5, color_list=None):

    if cluster_subset is not None:
        df = copy.copy(df)
        df = df[df[cluster_label].isin(cluster_subset)]
        
    timeseries_list = []
    for feature_label in feature_labels:        
        agg_cluster_timeseries = df.groupby([cluster_label]).apply(lambda x: agg_fn(np.array(x[feature_label].tolist()), axis=0)).to_frame()
        agg_cluster_timeseries = agg_cluster_timeseries.rename(columns={0:feature_label})
        timeseries_list.append(agg_cluster_timeseries)
    timeseries_df = pd.concat(timeseries_list, axis=1)
        
    fig = plt.figure(constrained_layout=True, figsize=figsize)
    gs = fig.add_gridspec(1,len(timeseries_df), wspace=wspace)

    for i in range(len(timeseries_df)):
        mean_cluster_timeseries = timeseries_df.iloc[i]
        
        clust_arr = np.array(timeseries_df.iloc[i].tolist())
        if color_list == None:
            color = "tab:blue"
        else:
            color = color_list[i%len(color_list)]
        
        if i == 0:
            inner_gs = gs[0,i].subgridspec(clust_arr.shape[0], 1, wspace=0, hspace=hspace)
            inner_grid_sub = inner_gs.subplots(sharex=True)
            for c, ax in np.ndenumerate(inner_grid_sub):
                
                feature_range = feature_range_list[c[0]]
                ax.plot(clust_arr[c],linewidth=linewidth,color=color)
                ax.set_ylim(feature_range[0],feature_range[1])
                ax.set(xticks=[])
#                 ax.set(xticks=[], yticks=[0,6])
                ax.set_ylabel(displayed_labels[c[0]],rotation=0,labelpad=30,fontsize=fontsize,ha="right")#,orientation="horizontal")
                
            ax.set_xlabel(str(i), fontsize=fontsize)
            ax.set(xticks=x_ticks)
            ax.set(xticklabels=x_ticklabels)
            
        else:
            inner_gs = gs[0,i].subgridspec(clust_arr.shape[0], 1, wspace=0, hspace=hspace)
            inner_grid_sub = inner_gs.subplots(sharex=True)
            for c, ax in np.ndenumerate(inner_grid_sub):
                feature_range = feature_range_list[c[0]]
                ax.plot(clust_arr[c],linewidth=linewidth,color=color)
                ax.set_ylim(feature_range[0],feature_range[1])
                ax.set(xticks=[], yticks=[])
                
            ax.set_xlabel(str(i), fontsize=fontsize)
            ax.set(xticks=x_ticks)
            ax.set(xticklabels=x_ticklabels)


    plt.tight_layout()
    return fig

In [ ]:
label_list = ['Kernel Trace: Delta time (hr)','Kernel Trace: Septum Displacement Length Normalized','Kernel Trace: major_axis_length',\
'Kernel Trace: minor_axis_length','Kernel Trace: mCherry mean_intensity','Kernel Trace: Instantaneous Growth Rate: Volume']
title_list = ["Interdivision Time (hr)","Septum Displacement/Length","Length $(\mu m)$","Width $(\mu m)$",\
              "mKate2 Mean Intensity (AU)","Growth Rate (1/hr)"]
range_list = []
for i,label in enumerate(label_list):
    vmin = np.percentile(np.array(an_df_clustree.obs[label].tolist()),0.1)
    vmax = np.percentile(np.array(an_df_clustree.obs[label].tolist()),99.9)
    range_list.append((vmin,vmax))

fig = plot_cluster_timeseries(an_df_clustree.obs,"L1", label_list,title_list,range_list,figsize=(20,6), x_ticks = [0,10,20], x_ticklabels=[0,5,10] ,wspace=0.25,hspace=0.25,color_list=vega_20_scanpy)
fig.savefig(umap_path + "/Leiden1_Cluster_Timeseries.png",dpi=300)

In [ ]:
label_list = ['Kernel Trace: Delta time (s): Transformed: z score','Kernel Trace: Septum Displacement Length Normalized: Transformed: z score','Kernel Trace: major_axis_length: Transformed: z score',\
'Kernel Trace: minor_axis_length: Transformed: z score','Kernel Trace: mCherry mean_intensity: Transformed: z score','Kernel Trace: Instantaneous Growth Rate: Volume: Transformed: z score']
title_list = ["Interdivision Time Z-score","Septum Displacement/Length Z-score","Length Z-score","Width Z-score",\
              "mKate2 Mean Intensity Z-score","Growth Rate Z-score"]
range_list = []
for i,label in enumerate(label_list):
    vmin = np.percentile(np.array(an_df_clustree.obs[label].tolist()),0.1)
    vmax = np.percentile(np.array(an_df_clustree.obs[label].tolist()),99.9)
    range_list.append((vmin,vmax))

fig = plot_cluster_timeseries(an_df_clustree.obs,"L1", label_list,title_list,range_list,figsize=(20,6), x_ticks = [0,10,20], x_ticklabels=[0,5,10], wspace=0.25,hspace=0.25,color_list=vega_20_scanpy)
fig.savefig(umap_path + "/Leiden1_Cluster_Timeseries_Zscores.png",dpi=300)

### Highlighting Genes of Interest
- Ignore

In [ ]:
%%capture
fig = highlight_regex(an_df_clustree_r_compat,["lgt","lnt","lolA","lolB","lolC","lolD","lolE","lspA"],match_mode="exact",highlight="each");

In [ ]:
fig

In [ ]:
%%capture
fig = highlight_go(an_df_clustree_r_compat, ns2assoc, obodag, inv_gene_to_id, "GO:0043039", go_mode = "each", fontsize=18)

In [ ]:
fig

### Setting Cluster Identifiers

In [ ]:
cluster_level = "L5"
cluster_label = "93"

### Highlighted Cluster UMAP
- Option for coloring

In [ ]:
%%capture
cluster_highlight_fig = highlight_cluster(an_df_clustree_r_compat,cluster_level,cluster_label,resolution_list,color="purple")
cluster_highlight_fig.axes[0].get_legend().remove()

In [ ]:
cluster_highlight_fig

In [ ]:
cluster_highlight_fig.savefig(selected_path+"/cluster_highlight_fig.png",dpi=150)

### Profile Box Plot

In [ ]:
## violin plot settings
labels = ['Delta time (hr)','Septum Displacement Length Normalized','major_axis_length',\
          'minor_axis_length','mCherry mean_intensity','Instantaneous Growth Rate: Volume']
kernel_labels = ['Kernel Trace: ' + label for label in labels]
label_names = ["Interdivision Time (hr)","Septum Displacement","Long Axis Length ($\mu m$)",\
          "Short Axis Length ($\mu m$)","mKate2 Intensity (AU)","Growth Rate (1/hr)"]
kernel_rename_labels = dict(zip(kernel_labels,label_names))
param_ranges = [(0,4.),(0.,0.5),(0.,14.),(1.,1.6),(0,10000),(0.,2.)]

def get_cluster_box(clustree_pandas_df,cluster_level,cluster_label,\
                       labels=labels,kernel_labels=kernel_labels,label_names=label_names,\
                       param_ranges=param_ranges,figsize=(14,3)):
    sns.set_style("ticks")
    
    kernel_rename_labels = dict(zip(kernel_labels,label_names))
    selected_df = clustree_pandas_df[clustree_pandas_df[cluster_level]==cluster_label]
    selected_df = selected_df.rename(columns=kernel_rename_labels)
    last_value_df = selected_df[label_names].apply(lambda x: [kernel[-1] for kernel in x.tolist()], axis=0).melt()
    first_value_df = selected_df[label_names].apply(lambda x: [kernel[0] for kernel in x.tolist()], axis=0).melt()
    
    fig = plt.figure(figsize=figsize)
    for i in range(len(kernel_labels)):
        plt.subplot(1,len(kernel_labels)*2,(i*2)+1)
        filtered_last_value_df = last_value_df[last_value_df["variable"]==label_names[i]].copy()
        filtered_first_value_df = first_value_df[first_value_df["variable"]==label_names[i]].copy()
        filtered_last_value_df.loc[:, ('variable',)] = "Last Timepoint"
        filtered_first_value_df.loc[:, ('variable',)] = "First Timepoint"
        merged_df = pd.concat([filtered_last_value_df,filtered_first_value_df]).reset_index()
        # filtered_last_value_df["control_variable"] = filtered_first_value_df["variable"]
        sns.boxplot(data=merged_df,x="variable",y="value",order=["First Timepoint","Last Timepoint"],fliersize=0)
        plt.ylabel(label_names[i])
        plt.xticks(ticks=[0,1],labels=["Pre","Post"])
        plt.ylim(param_ranges[i][0],param_ranges[i][1])
        plt.xlabel("",rotation=90)
        
    return fig

def plot_single_cluster_timeseries(clustree_pandas_df,cluster_level,cluster_label,n_hours_expt = 22,figsize = (24,12),fontsize = 16,nrows = 2,ncols = 3,\
                           labels = labels, label_names = label_names, param_ranges = param_ranges):

    labels_map = {i:label for i,label in enumerate(labels)}
    kernel_trace_labels = ["Kernel Trace: " + label for label in labels]
    kernel_trace_zscore_labels = ["Kernel Trace: " + label + ": Transformed: z score" for label in labels]
    #hack to replace hr with s for zscore
    kernel_trace_zscore_labels = ['Kernel Trace: Delta time (s): Transformed: z score'] + kernel_trace_zscore_labels[1:]

    selected_df = clustree_pandas_df[clustree_pandas_df[cluster_level] == cluster_label]
    timseries_arr = np.array(selected_df[kernel_trace_labels].values.tolist())
    timeseries_zscore_arr = np.array(selected_df[kernel_trace_zscore_labels].values.tolist())
    obs_grid_map = {i:time for i,time in enumerate(np.linspace(0,n_hours_expt,num=timseries_arr.shape[2]))}
    
    timseries_xarr = xr.DataArray(timseries_arr,dims=["sgRNA","Property","Time (hr)"])
    timseries_df = timseries_xarr.to_dataframe("Observable")
    timseries_df = timseries_df.reset_index()
    timseries_df["Property"] = timseries_df["Property"].apply(lambda x: labels_map[x])
    timseries_df["Time (hr)"] = timseries_df["Time (hr)"].apply(lambda x: obs_grid_map[x])    

    fig = plt.figure(figsize=figsize)
    for prop_idx in range(timseries_arr.shape[1]):
        plt.subplot(nrows,ncols,prop_idx+1)

        timseries_df_single_prop = timseries_df[timseries_df["Property"] == labels[prop_idx]]
        timseries_df_single_prop = timseries_df_single_prop.rename(columns={"Observable":labels[prop_idx]})
        sns.lineplot(x="Time (hr)",y=labels[prop_idx], data=timseries_df_single_prop, ci="sd")
        plt.ylim(param_ranges[prop_idx][0],param_ranges[prop_idx][1])
        plt.ylabel(label_names[prop_idx],fontsize=fontsize)
        plt.xlabel("Time (hr)",fontsize=fontsize)
        plt.xticks(fontsize=fontsize)
        plt.yticks(fontsize=fontsize)
    plt.tight_layout()
    return fig

In [ ]:
%%capture
boxplot_fig = get_cluster_box(clustree_pandas_df,cluster_level,cluster_label,\
                         labels=labels,kernel_labels=kernel_labels,label_names=label_names,\
                         param_ranges=param_ranges)

In [ ]:
boxplot_fig

In [ ]:
boxplot_fig.savefig(selected_path+"/boxplot_fig.png",dpi=200)

### Profile Timeseries

In [ ]:
%%capture
timeseries_fig = plot_single_cluster_timeseries(clustree_pandas_df,cluster_level,cluster_label,labels=labels,label_names=label_names,\
                         param_ranges=param_ranges,figsize=(10,6))

In [ ]:
timeseries_fig

In [ ]:
timeseries_fig.savefig(selected_path+"/timeseries_fig.png",dpi=170)

### GO Enrichment Analysis

In [ ]:
def wrap_text(input_string):
    space_positions = [match.start() for match in re.finditer(" ", input_string)]
    if len(space_positions)>0:
        total_length = len(input_string)
        min_space = np.argmin(abs(np.array(space_positions)-(total_length//2)))
        best_break_position = space_positions[min_space]
        input_string = input_string[:best_break_position] + "\n" + input_string[best_break_position+1:]
    return input_string

def get_GO_chart(clustree_pandas_df,cluster_level,cluster_label,obodag=obodag,objanno=objanno,ns2assoc=ns2assoc,min_gene_count=2,figsize=(2,6)):
    all_genes = clustree_pandas_df["Gene"].unique().tolist()
    selected_df = clustree_pandas_df[clustree_pandas_df[cluster_level]==cluster_label]
    cluster_genes,cluster_gene_counts = np.unique(selected_df["Gene"],return_counts=True)
    cluster_gene_list = sorted(cluster_genes[cluster_gene_counts>=min_gene_count].tolist())

    goea_quiet_enriched = get_enriched_GO_terms(all_genes,cluster_gene_list,obodag,objanno,ns2assoc,pval=0.05,GO_type="BP")
    filtered_go_terms = get_filtered_go_terms(obodag,objanno,goea_quiet_enriched,sim_thr = 0.3,info_thr = 1.)
    go_term_dict = {go_term.name:(go_term.ratio_in_study[0]/go_term.ratio_in_study[1]) for go_term in filtered_go_terms}

    go_term_keys = list(go_term_dict.keys())
    go_term_vals = [go_term_dict[key] for key in go_term_keys]
    go_term_keys = [key.title() for key in go_term_keys]
    go_term_keys = [wrap_text(key) for key in go_term_keys]
    
    # newline_position = [re.search("[a-zA-Z]{0,100} Process",key).start() if re.search("[a-zA-Z]{0,100} Process",key)!=None else -1 for key in go_term_keys]
    # go_term_keys = [key[:newline_position[i]] + "\n" + key[newline_position[i]:] for i,key in enumerate(go_term_keys)]
    
    fig, ax = plt.subplots(figsize=figsize)
    plt.bar(go_term_keys,go_term_vals)
    plt.ylim(0,1)
    plt.ylabel("Frequency of GO Term in Cluster")
    plt.xticks(rotation=90)
    plt.tight_layout()
    return fig

def get_gene_chart(clustree_pandas_df,cluster_level,cluster_label,min_gene_count=2,width=0.35,figsize=(2,6),padding=1.,genefontsize=12):

    selected_df = clustree_pandas_df[clustree_pandas_df[cluster_level]==cluster_label]
    gene_recall_series = get_recall_series(clustree_pandas_df,selected_df,recall_label="Gene")
    cluster_genes,cluster_gene_counts = np.unique(selected_df["Gene"],return_counts=True)

    total_genes = np.sum(cluster_gene_counts)
    total_under_min = np.sum(cluster_gene_counts[cluster_gene_counts<min_gene_count])

    cluster_gene_over_min = sorted(cluster_genes[cluster_gene_counts>=min_gene_count].tolist())
    cluster_gene_dict = {gene:cluster_gene_counts[i] for i,gene in enumerate(cluster_genes) if gene in cluster_gene_over_min}
    # cluster_gene_dict["Other"] = total_under_min

    cluster_gene_keys = list(cluster_gene_dict.keys())
    cluster_gene_vals = [cluster_gene_dict[key] for key in cluster_gene_keys]
    cluster_gene_recalls = gene_recall_series[cluster_gene_keys].tolist()
    cluster_gene_keys = [key.split()[0] + "..." if len(key.split())>1 else key for key in cluster_gene_keys]

    fig, ax1 = plt.subplots(figsize=figsize)
    ax2 = ax1.twinx() 

    tick_locations = np.array(list(range(len(cluster_gene_keys))))

    bar1 = ax1.bar(tick_locations-width/2,cluster_gene_vals,width=width,label="Counts",color="C0",align='edge')
    bar2 = ax2.bar(tick_locations+width/2,cluster_gene_recalls,width=width,label="Recall",color="C1",align='edge')
    ax1.set_xticks(tick_locations,labels=cluster_gene_keys,rotation=90,fontsize=genefontsize)
    
    # bars = bar1+bar2
    labs = [bar1.get_label(),bar2.get_label()]
    # ax2.legend([bar1,bar2], labs, loc=0)

    ax1.set_ylabel('Variant Count', color = 'C0')
    ax1.tick_params(axis ='y', labelcolor = 'C0')

    ax2.set_ylabel('Gene Recall', color = 'C1')
    ax2.set_ylim(0.,1.)
    ax2.tick_params(axis ='y', labelcolor = 'C1')
        
    plt.xlim((tick_locations-width/2)[0]-padding,(tick_locations+width/2)[-1]+padding)
    plt.tight_layout()
    return fig

def get_recall_series(clustree_pandas_df,selected_df,recall_label="TargetID"):
    recall_metric_count_dict = dict(zip(*np.unique(clustree_pandas_df[recall_label],return_counts=True)))
    all_counts = np.sum(list(recall_metric_count_dict.values()))

    recall_count_in_cluster = pd.DataFrame.from_dict(dict(zip(*np.unique(selected_df[recall_label],return_counts=True))),orient='index')
    recall_count_all_clusters = pd.DataFrame.from_dict({key:recall_metric_count_dict[key] for key in np.unique(selected_df[recall_label])},orient='index').rename(columns={0:1})
    recall_df = pd.concat([recall_count_in_cluster,recall_count_all_clusters],axis=1).rename(columns={0:"In Cluster",1:"All Clusters"})
    recall_series = recall_df["In Cluster"]/recall_df["All Clusters"]
    
    return recall_series

In [ ]:
%%capture
GO_fig = get_GO_chart(clustree_pandas_df,cluster_level,cluster_label,min_gene_count=2,figsize=(5,7))

In [ ]:
GO_fig

In [ ]:
GO_fig.savefig(selected_path+"/GO_fig.png",dpi=150)

### Gene Analysis

In [ ]:
# old = gene_analysis_fig.axes[0].get_xticklabels()
# old_label = old[7]
# old_label.set_text("insEF-1/2/3/4/5 insF-1/2/3/4")
# old[7] = old_label

In [ ]:
# old_label.set_text("insEF-1/2/3/4/5 insF-1/2/3/4")

In [ ]:
# %%capture
# gene_analysis_fig = get_gene_chart(clustree_pandas_df,cluster_level,cluster_label,min_gene_count=2,figsize=(4,3),mode="count")
gene_analysis_fig = get_gene_chart(clustree_pandas_df,cluster_level,cluster_label,min_gene_count=2,figsize=(4*4.1,3*1),padding=1.,genefontsize=12)
plt.tight_layout()
# ticklabels = gene_analysis_fig.axes[0].get_xticklabels()
# old_label = ticklabels[7]
# old_label.set_text("insEF-1/2/3/4/5\ninsF-1/2/3/4")
# ticklabels[7] = old_label
# gene_analysis_fig.axes[0].set_xticklabels(ticklabels)
# plt.tight_layout()

In [ ]:
gene_analysis_fig.savefig(selected_path+"/gene_analysis.png",dpi=200)

### Making Figure Canvas

In [ ]:
import sys
from PIL import Image

path_names = ["boxplot_fig.png","cluster_highlight_fig.png","GO_fig.png","timeseries_fig.png","gene_analysis.png"]
full_path_names = [selected_path+"/"+path for path in path_names]
coords = [(800,10),(10,10),(10,700),(1450,700),(10,1950)]
# creates a new empty image, RGB mode, and size 444 by 95
new_im = Image.new('RGB', (3300,2550), color=(255,255,255))

for i,full_path in enumerate(full_path_names):
    im=Image.open(full_path)
    new_im.paste(im, coords[i])
new_im

In [ ]:
new_im.save(selected_path+'/' + str(cluster_level) + "_" + str(cluster_label) + ".png")

### Zip Current Clustree Folder

In [ ]:
import shutil

rootpath = "/home/de64/group/CRISPRi_Libraries/dev_notebooks/2023-02-26_lDE20_10Hour_Analysis"
dir_above = "/".join(rootpath.split("/")[:-1]) + "/"
rootpath_dir = rootpath.split("/")[-1]
shutil.make_archive(rootpath,"zip",root_dir=dir_above,base_dir=rootpath_dir)